# **Implement Federated Learning for NLP (Text Classification)**
**Objective**

Build a Federated Learning system for NLP sentiment classification where:

1. Multiple clients train local models

2. Raw data never leaves the client

3. Server aggregates models using Federated Averaging

**Step 1 – Install Required Libraries (Colab)**

In [ ]:
!pip install torch torchvision torchtext

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 23.5 MB/s eta 0:00:00


**Step 2 – Import Libraries**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import Counter

**Step 3 – Sample NLP Dataset**

Example sentiment dataset.

In [ ]:
data = [
("I love this movie",1),
("This film is amazing",1),
("Fantastic acting and story",1),
("Great direction",1),
("Wonderful experience",1),
("I enjoyed the movie",1),

("I hate this movie",0),
("This film is terrible",0),
("Worst movie ever",0),
("Very boring film",0),
("Bad acting",0),
("I dislike this movie",0)
]

**Step 4 – Text Preprocessing**

Build vocabulary and convert text to numbers.

In [ ]:
def tokenize(text):
    return text.lower().split()

vocab = Counter()

for text,label in data:
    vocab.update(tokenize(text))

word2idx = {word:i+1 for i,(word,_) in enumerate(vocab.items())}

In [ ]:
#Convert sentences to vectors.

def encode(text):

    tokens = tokenize(text)

    return [word2idx[word] for word in tokens if word in word2idx]

**Step 5 – Simulate Federated Clients**

Split dataset into clients.

In [ ]:
NUM_CLIENTS = 3

random.shuffle(data)

client_data = []

size = len(data)//NUM_CLIENTS

for i in range(NUM_CLIENTS):

    start = i*size
    end = start+size

    client_data.append(data[start:end])

**Step 6 – Define NLP Model**

We use Embedding + Average + Linear classifier.

In [ ]:
class TextModel(nn.Module):

    def __init__(self,vocab_size):

        super(TextModel,self).__init__()

        self.embedding = nn.Embedding(vocab_size+1,50)

        self.fc = nn.Linear(50,2)

    def forward(self,x):

        embeds = self.embedding(x)

        avg = embeds.mean(dim=1)

        out = self.fc(avg)

        return out

**Step 7 – Prepare Training Data**

In [ ]:
def prepare_batch(dataset):

    X=[]
    y=[]

    for text,label in dataset:

        X.append(torch.tensor(encode(text)))
        y.append(label)

    return X,y

**Step 8 – Local Client Training**

In [ ]:
def train_client(model,data,epochs=2):

    X,y = prepare_batch(data)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(),lr=0.01)

    model.train()

    for epoch in range(epochs):

        for i in range(len(X)):

            optimizer.zero_grad()

            output = model(X[i].unsqueeze(0))

            loss = criterion(output,torch.tensor([y[i]]))

            loss.backward()

            optimizer.step()

    return model.state_dict()

**Step 9 – Federated Averaging**

In [ ]:
def federated_average(global_model,client_weights):

    global_dict = global_model.state_dict()

    for key in global_dict.keys():

        global_dict[key] = torch.stack(
            [client_weights[i][key] for i in range(len(client_weights))]
        ).mean(0)

    global_model.load_state_dict(global_dict)

    return global_model

**Step 10 – Federated Training**

In [ ]:
global_model = TextModel(len(word2idx))

ROUNDS = 5

for round in range(ROUNDS):

    client_weights=[]

    for client in range(NUM_CLIENTS):

        local_model = TextModel(len(word2idx))

        local_model.load_state_dict(global_model.state_dict())

        weights = train_client(local_model,client_data[client])

        client_weights.append(weights)

    global_model = federated_average(global_model,client_weights)

    print("Round",round,"completed")

Round 0 completed
Round 1 completed
Round 2 completed
Round 3 completed
Round 4 completed


**Step 11 – Test Prediction**

In [ ]:
def predict(text):

    global_model.eval()

    x = torch.tensor(encode(text)).unsqueeze(0)

    with torch.no_grad():

        output = global_model(x)

        probs = torch.softmax(output,dim=1)

        pred = torch.argmax(probs)

    return "Positive" if pred==1 else "Negative"

In [ ]:
print(predict("great movie"))
print(predict("terrible film"))

Positive
Negative
